# 9. MCP (Model Context Protocol)

## 학습 목표

1. MCP의 개념과 Host / Client / Server 구조를 설명할 수 있습니다.
2. `fastmcp`로 MCP Server를 만듭니다.
3. LangChain Agent(MCP Host)가 MCP Server의 도구를 불러와 실행하게 합니다.
4. "JSON 파일 읽기 → 요약 → Notion 기록"을 자연어 한 문장으로 자동 실행합니다.

**기준 버전**: LangChain 1.x, `langchain-mcp-adapters`, `fastmcp`, 모델 `gpt-6-luna`

## 1. MCP란?

- **정의**: LLM이 외부 데이터와 시스템을 표준화된 방식으로 활용하도록 설계된 **연결 프로토콜**입니다. Anthropic이 2024년 11월에 공개했습니다.
- **비유 - USB-C 포트**: USB-C가 다양한 기기를 하나의 규격으로 연결하듯, MCP는 AI 모델을 다양한 데이터 소스와 도구에 하나의 규격으로 연결합니다.

### 기존 LLM의 한계

- **텍스트 중심**: 주로 문장 생성과 답변에 치우쳐 있습니다.
- **격리된 환경**: 로컬 파일을 직접 다룰 수 없습니다.
- **단절된 기능**: 외부 서비스(API, SaaS)를 스스로 호출하지 못합니다.

→ 즉, 기존 LLM은 **"읽고 말하는 도구"** 에 머물러 있었습니다.

### MCP를 통한 구조적 확장

1. **사용자**: 자연어로 필요한 작업을 요청합니다.
2. **LLM(뇌)**: 요청을 분석해 어떤 "행동"이 필요한지 판단합니다.
3. **MCP Server(손과 발)**: 파일 접근, API 호출 같은 실제 실행을 담당합니다.
4. **피드백**: 실행 결과를 다시 LLM의 맥락에 넣어 최종 답변을 완성합니다.

> "MCP는 LLM에게 **생각을 넘어 행동을 실행할 수 있는 표준 인터페이스**를 제공합니다."

## 2. MCP의 3가지 핵심 구성 요소

**"누가 시키고(Host), 누가 연결하고(Client), 누가 일하는지(Server)"** 만 기억하면 됩니다.

| 구성 요소 | 비유 | 역할 | 예시 |
|---|---|---|---|
| MCP Host | 명령하는 **두뇌** (주방장) | 사용자와 소통하는 애플리케이션. LLM으로 요청을 분석하고 필요한 도구를 판단 | Claude Desktop, Cursor, 우리가 만드는 LangChain Agent |
| MCP Client | 연결하는 **신경망** (웨이터) | Host의 명령을 MCP 규약으로 Server에 전달하고 결과를 가져옴. Server와 1:1 연결 | `mcp` 공식 SDK, `langchain-mcp-adapters` |
| MCP Server | 일하는 **손발** (조리 도구) | 파일 읽기, DB 저장, 검색, 외부 API 호출 등 실제 작업 수행 | 우리가 만드는 `mcp_server.py` |

### 핵심 원칙: "LLM은 직접 실행하지 않는다"

```text
LLM(Host)이 판단 → Client가 요청 전달 → Server가 실행 → 결과가 Host로 돌아옴
```

참고 자료: [Visual Guide to Model Context Protocol (MCP)](https://www.dailydoseofds.com/p/visual-guide-to-model-context-protocol-mcp/)

## 3. fastmcp란?

`fastmcp`는 MCP Server를 가장 간단하게 만들 수 있는 Python 라이브러리입니다.

- Python 함수 하나 = MCP Tool 하나 (`@mcp.tool` 데코레이터)
- Tool 설명(스키마)을 타입 힌트와 Docstring으로 자동 생성
- STDIO 방식 실행 지원 (로컬 실습에 적합)

| 구현 대상 | 사용 라이브러리 |
|---|---|
| MCP Server | `fastmcp` |
| MCP Client + Host | `langchain-mcp-adapters` + LangChain `create_agent` |

## 4. 실습: 사용자 질문 → 실험 결과(JSON) → Notion 자동 기록

사용자가 이렇게 요청합니다.

> "train_result.json 파일을 요약해서 Notion에 기록해 줘."

이 한 문장으로 다음 작업이 자동 실행됩니다.

1. 로컬 실험 결과 JSON 파일 읽기
2. LLM이 실험 결과를 이해하고 요약
3. 요약 결과를 Notion 페이지로 기록

> **핵심**: 자연어 → LLM 판단 → MCP Tool 실행 → 실제 외부 시스템 반영. 이 한 사이클을 직접 완주합니다.

```text
mcp_host.py (Host + Client, LangChain Agent)
     │  stdio로 서버 프로세스를 직접 실행·연결
     ▼
mcp_server.py (MCP Server, fastmcp)
     ├─ read_experiment_result  → train_result.json 읽기
     └─ upload_experiment_to_notion → Notion 페이지 생성
```

### 실습 환경 준비

터미널에서 필요한 패키지를 설치하고, 프로젝트 최상위 폴더의 `.env` 파일에 API 키, 모델명, Notion 정보를 저장합니다.

```text
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
OPENAI_DEFAULT_MODEL="gpt-6-luna"
NOTION_TOKEN="Notion 통합 토큰"
NOTION_PAGE_ID="기록할 Notion 페이지 ID(32자리)"
```

> API 키는 코드 셀이나 GitHub에 직접 입력하지 마세요.

```bash
uv add langchain langchain-openai langgraph langchain-mcp-adapters fastmcp notion-client python-dotenv
```

`NOTION_TOKEN`, `NOTION_PAGE_ID` 만드는 방법은 5장에서 설명합니다.

> **gpt-6-luna 사용 시 참고**
> - `reasoning_effort`는 `none` / `low` / `medium`(기본) / `high` / `xhigh` / `max`를 지원합니다. 강의 예제는 `none`으로 통일합니다.
> - `temperature`는 추론이 꺼져 있을 때(`reasoning_effort="none"`)만 적용됩니다.
> - 모델명·요금·지원 파라미터는 바뀔 수 있으므로 수업 전에 [OpenAI 모델 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)를 확인하세요.

## 5. Notion 토큰과 페이지 권한 준비

### 5-1. Notion 통합 토큰 만들기 (NOTION_TOKEN)

1. Notion에 로그인한 뒤 [개발자 포털(https://app.notion.com/developers/connections)](https://app.notion.com/developers/connections)에 접속합니다.
   (또는 Notion 설정 → 연결 → **개발자 포털로 이동**)
2. **+ New integration** 을 클릭합니다.
3. 이름(예: `FastMCP`, `MCP Server`. "Notion"이라는 단어는 이름에 쓸 수 없음)과 워크스페이스를 선택하고 생성합니다.
4. 생성된 통합의 상세 화면에서 **API 통합 토큰**을 표시(Show)하고 복사합니다. → `NOTION_TOKEN`

### 5-2. 페이지 접근 권한 부여하기 (NOTION_PAGE_ID)

- 토큰은 워크스페이스 전체가 아니라 **연결을 허용한 페이지/DB만** 접근할 수 있습니다.
- 통합 설정의 **사용 권한(Access)** 탭에서 MCP로 기록할 페이지를 선택합니다.
- 해당 페이지 URL의 마지막 **32자리**가 Page ID입니다. `.env`에 `NOTION_PAGE_ID`로 저장합니다.

URL에서 Page ID를 추출하는 코드입니다.

In [ ]:
import re

url = "https://www.notion.so/My-Page-23a1b7c4d5e64f2b8a9c123456789abc"   # 나의 노션 페이지 URL

# ?뒤의 파라미터를 버리고, 하이픈을 지운 뒤 끝의 32자리를 추출합니다.
match = re.search(r"([a-f0-9]{32})$", url.split("?")[0].replace("-", ""))
if match:
    print(match.group(1))

## 6. 더미 실험 결과 파일 준비

노트북과 같은 폴더에 `train_result.json` 파일을 만듭니다.

In [ ]:
%%writefile train_result.json
{
  "epoch": 0.12,
  "global_step": 500,
  "loss": 0.3358,
  "learning_rate": 1.8812351543942996e-05,
  "total_flos": 265988832528000
}

## 7. MCP Server 구현 - `mcp_server.py` (fastmcp)

- `@mcp.tool`을 붙인 함수 2개가 MCP Tool이 됩니다.
- STDIO 방식에서는 **표준 출력(stdout)이 MCP 통신 통로**이므로, 디버깅 메시지는 반드시 `stderr`로 출력합니다.

In [ ]:
%%writefile mcp_server.py
import json
import os
import sys

from dotenv import load_dotenv
from fastmcp import FastMCP
from notion_client import Client

load_dotenv()

mcp = FastMCP("ExperimentResultServer")

NOTION_TOKEN = os.getenv("NOTION_TOKEN")
NOTION_PAGE_ID = os.getenv("NOTION_PAGE_ID")

if not NOTION_TOKEN or not NOTION_PAGE_ID:
    print("Warning: NOTION_TOKEN 또는 NOTION_PAGE_ID가 없습니다.", file=sys.stderr)

notion = Client(auth=NOTION_TOKEN)


@mcp.tool
def read_experiment_result(file_path: str) -> str:
    """모델 학습 결과 JSON 파일을 읽어 문자열로 반환합니다.

    Args:
        file_path: JSON 파일 경로
    """
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"File read: {file_path}", file=sys.stderr)
        return json.dumps(data, ensure_ascii=False, indent=2)
    except Exception as e:
        return f"파일을 읽는 중 오류가 발생했습니다: {e}"


@mcp.tool
def upload_experiment_to_notion(title: str, summary: str) -> str:
    """요약된 실험 결과를 Notion 페이지로 업로드합니다.

    Args:
        title: 페이지 제목
        summary: 실험 결과 요약
    """
    try:
        notion.pages.create(
            parent={"page_id": NOTION_PAGE_ID},
            properties={"title": {"title": [{"text": {"content": title}}]}},
            children=[
                {
                    "object": "block",
                    "type": "paragraph",
                    "paragraph": {"rich_text": [{"type": "text", "text": {"content": summary}}]},
                }
            ],
        )
        print(f"Uploaded to Notion: {title}", file=sys.stderr)
        return "Notion 업로드 완료"
    except Exception as e:
        return f"Notion 업로드 중 오류가 발생했습니다: {e}"


if __name__ == "__main__":
    mcp.run(transport="stdio")

## 8. MCP Host + Client 구현 - `mcp_host.py`

- `MultiServerMCPClient`(MCP Client)가 `mcp_server.py`를 **자식 프로세스로 실행하고 stdio로 연결**합니다.
- `client.get_tools()`로 MCP Server의 도구를 LangChain Tool로 불러옵니다.
- `create_agent`(MCP Host의 두뇌)가 도구를 단계별로 사용해 요청을 처리합니다.

In [ ]:
%%writefile mcp_host.py
import asyncio
import os
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

load_dotenv()


async def main():
    # 1. MCP Client: 서버 프로세스를 실행하고 연결
    client = MultiServerMCPClient({
        "experiment": {
            "transport": "stdio",
            "command": sys.executable,   # 현재 가상환경의 python
            "args": ["mcp_server.py"],
        }
    })

    # 2. MCP Server의 도구 불러오기
    tools = await client.get_tools()
    print(f"Tool: {[t.name for t in tools]}\n")

    # 3. LLM + Agent (MCP Host)
    llm = ChatOpenAI(model=os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna"), reasoning_effort="none")
    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=(
            "너는 MCP Tool 전문가야. 사용자의 요청을 정확히 처리해.\n"
            "1. read_experiment_result로 JSON 파일을 읽고\n"
            "2. 내용을 요약한 뒤\n"
            "3. upload_experiment_to_notion으로 Notion에 업로드해."
        ),
    )

    # 4. 사용자 질문
    user_question = (
        "train_result.json 파일을 요약해서 Notion에 기록해 줘. "
        "제목은 '모델 실험 결과 - step 500'으로 해 줘."
    )
    result = await agent.ainvoke({"messages": [("human", user_question)]})

    # 5. 결과 출력
    for msg in result["messages"]:
        if msg.type == "ai" and msg.tool_calls:
            print(f"[도구 요청] {[c['name'] for c in msg.tool_calls]}")
    print(f"\n결과: {result['messages'][-1].text}")


if __name__ == "__main__":
    asyncio.run(main())

## 9. 실행하기

`mcp_host.py`만 실행하면 됩니다. **STDIO 방식에서는 Host가 Server 프로세스를 직접 실행**하므로, 서버를 다른 창에서 미리 켜 둘 필요가 없습니다.

터미널(PowerShell 또는 CMD)에서:

```bash
uv run python mcp_host.py
```

노트북에서 바로 실행하려면 아래 셀을 실행합니다.

> Windows의 Jupyter 환경에서는 노트북 안에서 stdio 서버를 직접 띄우는 비동기 코드가 오류를 낼 수 있어, 별도 파이썬 프로세스로 실행합니다.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "mcp_host.py"], capture_output=True, text=True, encoding="utf-8")
print(result.stdout)
print(result.stderr[-2000:])   # 서버 로그(stderr) 마지막 부분

### 실행 결과 확인

- 출력에 `[도구 요청] ['read_experiment_result']` → `[도구 요청] ['upload_experiment_to_notion']` 순서가 보이면 Agent가 도구를 단계별로 사용한 것입니다.
- Notion 페이지 아래에 **"모델 실험 결과 - step 500"** 하위 페이지가 생성되었는지 확인합니다.

### 오류가 날 때 확인할 것

| 증상 | 확인 사항 |
|---|---|
| Notion 업로드 오류 (401/403) | `NOTION_TOKEN` 값, 통합에 페이지 접근 권한을 줬는지 |
| Notion 업로드 오류 (404) | `NOTION_PAGE_ID`가 32자리 ID인지, 권한을 준 페이지인지 |
| 파일을 읽을 수 없음 | `train_result.json`이 `mcp_host.py`와 같은 폴더에 있는지 |
| 서버 연결 오류 | `mcp_server.py`를 단독 실행(`uv run python mcp_server.py`)해 import 오류가 없는지 확인 (확인 후 `Ctrl+C`로 종료) |